In [42]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,text,label
0,1,i didnt feel humiliated,sadness
1,2,i can go from feeling so hopeless to so damned...,sadness
2,3,im grabbing a minute to post i feel greedy wrong,anger
3,4,i am ever feeling nostalgic about the fireplac...,love
4,5,i am feeling grouchy,anger


In [43]:
df_train.text[46]

'i lost my special mind but don t worry i m still sane i just wanted you to feel what i felt while reading this book i don t know how many times it was said that sam was special but i can guarantee you it was many more times than what i used in that paragraph did i tell you she was special'

In [44]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', "", text)
    text = re.sub(r'[\d+]', "", text)
    return text

preprocess(df_train.text[46])

'i lost my special mind but don t worry i m still sane i just wanted you to feel what i felt while reading this book i don t know how many times it was said that sam was special but i can guarantee you it was many more times than what i used in that paragraph did i tell you she was special'

In [45]:
df_train.text = df_train.text.apply(preprocess)
print("train set done")
df_test.text = df_test.text.apply(preprocess)

df_train.head()

train set done


,SampleID,text,label
0,1,i didnt feel humiliated,sadness
1,2,i can go from feeling so hopeless to so damned...,sadness
2,3,im grabbing a minute to post i feel greedy wrong,anger
3,4,i am ever feeling nostalgic about the fireplac...,love
4,5,i am feeling grouchy,anger


In [46]:
df_train.label.unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [63]:
X = df_train.text
y = df_train.label
X_final = df_test.text

from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
y = enc.fit_transform(y)

y[:5]

array([4, 4, 0, 3, 0])

In [64]:
from sklearn.feature_extraction.text import CountVectorizer

all_text = pd.concat([df_train.text, df_test.text])
vect = CountVectorizer()
vect.fit(all_text)
X = vect.transform(X)
X_final = vect.transform(X_final)

X

<16000x16158 sparse matrix of type '<class 'numpy.int64'>'
	with 249634 stored elements in Compressed Sparse Row format>

In [65]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

model = LinearSVC(max_iter=10000)

cross_val_score(model, X, y, cv=2, scoring='f1_weighted')

array([0.87542524, 0.87762093])

In [66]:
model.fit(X, y)
y_final = model.predict(X_final)
y_final[:5]

array([4, 4, 4, 2, 4])

In [67]:
y_final = enc.inverse_transform(y_final)
y_final[:5]

array(['sadness', 'sadness', 'sadness', 'joy', 'sadness'], dtype=object)

In [68]:
submission = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "label": y_final
})

submission

,SampleID,label
0,16001,sadness
1,16002,sadness
2,16003,sadness
3,16004,joy
4,16005,sadness
...,...,...
1995,17996,anger
1996,17997,anger
1997,17998,joy
1998,17999,joy


In [69]:
submission.to_csv("submission.csv", index=False)